# 33 — Migrating from `midas-integrate` (v1) to `midas-integrate-v2`

If you have a working v1 pipeline (paramstest.txt + `DetectorMapper` + `Map.bin` + the CSR hot-path) and want to use v2, this notebook shows the minimum-friction migration path. You do **not** have to rewrite anything immediately — v2 is designed to coexist with v1, share its binary artefacts (`Map.bin`, `residual_corr.bin`), and accept v1 paramstest files directly.

The migration is incremental:

1. **Day 0** — Keep using v1 for production batch integration. Use v2 only for calibration refinement (gradients!) and one-off analysis notebooks.
2. **Day 7** — Switch new pipelines to v2; keep old pipelines on v1 until they break.
3. **Day 30+** — Migrate the production hot-path to `kernels.integrate` once you've validated the binning kernel matches v1 to bit-level on your data.

## What stays the same

* `Map.bin` — wire format is unchanged; v2's `MapCache` reads v1's binary.
* `residual_corr.bin` — same. Bit-identical exchange across all three packages.
* All physical units (µm, degrees, Å) — no convention flip.
* Polarization / solid angle math.

## What changes

| concept | v1 | v2 |
|---|---|---|
| Parameter container | `IntegrationParams` (Python floats) | `IntegrationSpec` (torch tensors for refinable fields) |
| Forward model | `pixel_to_REta` (numpy) | `pixel_to_REta_from_spec` (torch, autograd-clean) |
| Binning entry | `detector_mapper.build_map` + `fused_csr` | `binning.{hard,soft,subpixel,polygon}` |
| Map persistence | `Map.bin` (lazy build, cached) | `MapCache` (wraps v1's `Map.bin`) |
| Residual map | `residual_corr.load_residual_correction_map` | `corrections.RBFResidualCorrection` (or `load_residual_corr_bin` from `midas_calibrate_v2`) |
| Output writers | `exporters` (xye, dat) | `io.writers` (xye, dat, fxye, esg, csv, 2d_csv, h5) |
| Peak fitting | `peakfit` / `peak_io` | none in v2 (use `midas_peakfit` directly) |
| Server | `server.py` (CLI batch) | `streaming.integrate_stream` (Python-native) |

In [ ]:
import os; os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
import numpy as np, torch

## 1. Load your existing v1 paramstest into a v2 spec

Zero code changes on the v1 side — `spec_from_v1_paramstest` reads the same text file your v1 pipeline already consumes. It also picks up the v2-introduced hex-lattice extension keys (`PixelLattice`, `Apothem`, `LatticeOrientation`) without breaking on older files that don't have them.

In [ ]:
from midas_integrate_v2.compat import spec_from_v1_paramstest

# spec = spec_from_v1_paramstest('/path/to/your/paramstest.txt')
# Tensors come back with dtype=float64, on CPU, requires_grad=False.
#
# To make geometry refinable for an autograd-based refinement:
# spec = spec_from_v1_paramstest('paramstest.txt', requires_grad=True)
print('see midas_integrate_v2.compat.spec_from_v1_paramstest signature')

## 2. Convert an in-memory `IntegrationParams` (v1) → `IntegrationSpec` (v2)

Use this when your existing code already has a v1 `IntegrationParams` dataclass in memory (loaded by `midas_integrate.params.parse_params`).

In [ ]:
from midas_integrate.params import IntegrationParams
from midas_integrate_v2.compat import spec_from_v1_params

# v1 side — pretend we loaded from your existing paramstest.
p = IntegrationParams()
p.NrPixelsY = 2048; p.NrPixelsZ = 2048
p.pxY = 200.0; p.pxZ = 200.0
p.Lsd = 940710.0; p.BC_y = 1024.7; p.BC_z = 994.3
p.tx = -0.08; p.ty = -0.26; p.tz = 0.01
p.Wavelength = 0.189714

# v2 side — drops in as a spec, all refinable fields are torch tensors.
spec = spec_from_v1_params(p)
print(type(spec.Lsd).__name__, '=', spec.Lsd)

## 3. Round-trip the other way: v2 spec → v1 paramstest

After refining a spec in v2 (`midas_calibrate_v2.calibrate(...)`), you often want to write it back as a v1 paramstest for downstream C tools that haven't been ported. `v1_params_from_spec` strips the autograd machinery and returns a plain `IntegrationParams`, which you can serialise via v1's existing writer.

In [ ]:
from midas_integrate_v2.compat import v1_params_from_spec

p_back = v1_params_from_spec(spec)
print('round-trip diff:')
for attr in ('Lsd', 'BC_y', 'BC_z', 'tx', 'ty', 'tz', 'Wavelength'):
    a = getattr(p, attr); b = getattr(p_back, attr)
    print(f'  {attr:12s}  v1={a!r:12s}  back={b!r:12s}  diff={b-a:+.2e}')

## 4. Reuse your existing `Map.bin`

v1's `Map.bin` is the **expensive** part of v1 — building it from scratch takes 10-60 s on a 2880² detector. v2 reads the same file via `MapCache` so you don't repay that cost when switching pipelines.

```python
from midas_integrate_v2 import MapCache
cache = MapCache('/path/to/your/Map.bin')
# Pass cache to any binner that supports cached maps.
```

If you need to write a fresh `Map.bin` from a v2 spec (e.g. for a brand-new detector geometry that v1 doesn't have a cache for):

```python
from midas_integrate_v2.binning import write_map_bin_from_geometry
write_map_bin_from_geometry(geom, '/output/Map.bin')
```

Same wire format as v1; v1's `DetectorMapper` reads it without modification.

## 5. Reuse your existing `residual_corr.bin`

Same story: same binary, all three packages read/write it. If you have a v1 residual map from `CalibrantIntegratorOMP --fit-residual-map 1`, plug it straight into v2:

```python
# v2 (torch, autograd-clean):
from midas_calibrate_v2.forward.residual_corr import load_residual_corr_bin
corr_map_tensor = load_residual_corr_bin('residual_corr.bin',
                                          NrPixelsY=2048, NrPixelsZ=2048)

# v1 (numpy):
from midas_integrate.residual_corr import load_residual_correction_map
corr = load_residual_correction_map('residual_corr.bin',
                                      NrPixelsY=2048, NrPixelsZ=2048)
```

Both load the same bytes. The torch path is differentiable through the lookup; the numpy path is the v1 production helper.

## 6. Mapping the v1 pipeline to v2

| v1 production step | v2 equivalent |
|---|---|
| `params = parse_params('paramstest.txt')` | `spec = spec_from_v1_paramstest('paramstest.txt')` |
| `mapper = DetectorMapper(params)` <br> `mapper.build_or_load('Map.bin')` | `geom = HardBinGeometry.from_spec(spec)` <br> *or* `MapCache('Map.bin')` |
| `result = fused_csr.integrate(image, mapper)` | `result = integrate_hard(image, geom)` |
| `corr_map = load_residual_correction_map(...)` | (in v2 the residual map flows through `IntegrationSpec.ResidualCorrectionMap` automatically) |
| `pol_factor = polarization_factor(...)` | `corrections.PolarizationCorrection(P_horizontal=...)` |
| `exporters.write_xye(profile, ...)` | `io.write_xye(profile, sigma=..., metadata=...)` |
| `peakfit.fit_peak(profile, ...)` | *(use `midas_peakfit` directly — v2 doesn't ship its own peakfit)* |
| `server.py` batch CLI | `streaming.integrate_stream(frame_source, spec)` |

## 7. Common gotchas

### 7a. Tensor vs scalar

v2 refinable fields are `torch.Tensor`s, not Python floats. If you pass them into `print()` or `f'{x:.3f}'` they format fine, but if you pass them into C-extension code that expects a float, wrap with `float(...)`:

```python
# WRONG:
some_c_func(spec.Lsd)
# Right:
some_c_func(float(spec.Lsd))
```

Non-refinable fields (`NrPixelsY`, `pxY`, `RhoD`) are plain Python ints/floats — no wrap needed.

### 7b. dtype

v2 defaults to `torch.float64`. v1's residual-correction binary is `float64`. They match. If you switch v2 to `float32` (mobile-class hardware), be aware that the C tools still write `float64` — you'll need an explicit cast at the boundary.

### 7c. BC convention

Both v1 and v2 use **pixel coordinates** for `BC_y`, `BC_z`. No flip. `(BC_y, BC_z)` is `(column, row)` in image-array indexing.

### 7d. RhoD units

Both packages store `RhoD` in **pixels**, despite legacy v1 paramstest comments sometimes saying "µm". The v2 `forward.geometry.pixel_to_REta` does the px→µm conversion internally; v1's C path expects µm and the `IntegrationParams` field is converted by the C caller. If you're seeing distortion blow up, the RhoD sanity check in `midas_calibrate_v2.forward.sanity.check_rho_d_um` will catch it.

### 7e. `Map.bin` invalidation

If you change `BC`, `Lsd`, `tilts`, or any distortion coefficient, your cached `Map.bin` is stale. v2's `MapCache` *does not* validate this against the current spec — delete the file when geometry changes. (The cleanest pattern: cache one Map.bin per calibration, named with the calibration's hash.)

## 8. The drop-in replacement recipe

If you have an existing v1 script:

```python
# v1 production script
from midas_integrate import (
    IntegrationParams, parse_params,
    DetectorMapper, fused_csr,
    exporters,
)
params = parse_params('paramstest.txt')
mapper = DetectorMapper(params)
mapper.build_or_load('Map.bin')
profile = fused_csr.integrate(image, mapper)
exporters.write_xye(profile, 'out.xye')
```

Drop-in v2 equivalent:

```python
# v2 production script
from midas_integrate_v2 import (
    spec_from_v1_paramstest,
    HardBinGeometry, integrate_hard,
    write_xye,
)
spec = spec_from_v1_paramstest('paramstest.txt')
geom = HardBinGeometry.from_spec(spec)
profile = integrate_hard(image, geom)
write_xye('out.xye', profile.R_centres, profile.intensity)
```

Same paramstest, same `Map.bin` (built by v1 long ago, or built by v2 via `write_map_bin_from_geometry`), same output format. Run both side-by-side and diff the XYE outputs — they should agree at the 0.1% level.

If the diff is bigger, see NB 21 (pyFAI handoff) for the bin-centre vs bin-edge convention check, or run NB 25's `integrate_with_corrections` to make sure both pipelines apply the same physics corrections.

## 9. When to stay on v1

* **Frozen production pipelines** validated against published results — don't migrate just to migrate.
* **Pure batch integration** with no need for gradients — v1's `fused_csr` is the fastest path; v2's `integrate_hard` is close but not yet identical in microbenchmark.
* **Embedded / OS-restricted environments** — v1 is numpy + numba; v2 is torch + numpy + numba. Torch's footprint is large.
* **Peak-fitting in the same script** — `midas_peakfit` works with both, but v1's `peakfit.py` shim is more convenient if you're already using v1 everywhere else.

## When to move to v2

* **Anything that needs a gradient** — calibration refinement, learnable masks/gain, joint optimisation across frames, drift trajectory fitting.
* **New code** — v2 is the active development target; new features land here first.
* **Anywhere you previously hand-rolled outlier rejection / Q-uniform binning / corrections** — v2 ships modules for all of these.
* **PDF / Rietveld pipelines** — v2's `pdf` module + variance propagation is much cleaner than v1's manual approach.